# 06 · Eventos del Catálogo

Este notebook aplica los eventos del catálogo definidos en `eventos_catalogo.csv`.

Objetivos:
- Cargar los eventos del catálogo.
- Aplicar operaciones UPSERT y DELETE.
- Garantizar idempotencia mediante `events_log`.
- Sincronizar SQLite y Qdrant.
- Verificar visibilidad de productos tras cada evento.
- Demostrar que el sistema responde correctamente a mutaciones del catálogo.

Este notebook utiliza:
- `ingestion.py` (SQLite)
- `vector_ingestion.py` (Qdrant)
- `events.py` (aplicación de eventos)
- `search_engine.py` (verificación vectorial)


#### Importar librerías

In [1]:
import sys
import os

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

print("Ruta añadida al PYTHONPATH:", ROOT_DIR)


Ruta añadida al PYTHONPATH: /home/alexd/modulo_vector_bbdd/actividad_evaluable


In [2]:
import pandas as pd

from src.utils import safe_read_csv, log_section
from src.events import apply_catalog_events, apply_event, verify_visibility
from src.search_engine import search
from src.ingestion import get_connection


#### Cargar catálogo y eventos

In [3]:
log_section("Cargar catálogo y eventos")

df_catalog = safe_read_csv("../data/catalogo_muestra.csv")
df_events = safe_read_csv("../data/eventos_catalogo.csv")

df_events.head()


[AURUM] 
[AURUM] ============================================================
[AURUM] Cargar catálogo y eventos
[AURUM] ============================================================
[AURUM] [CSV] Cargado: ../data/catalogo_muestra.csv (1500 filas)
[AURUM] [CSV] Cargado: ../data/eventos_catalogo.csv (24 filas)


,sequence,event_id,operation,record_id,product_id,title,brand,color,locale,text,catalog_version,active
0,1,EVT-001,UPSERT,e1a0e559-6a49-5be5-b617-ec8a4899e975,B000G3T55M,NIKE Legasee Legging Swoosh Pantalones Deporti...,NIKE,Negro (Black/White 011),es,NIKE Legasee Legging Swoosh Pantalones Deporti...,2,True
1,2,EVT-002,UPSERT,0df8a596-0bc2-5deb-bc84-69b63972f975,B07NV4L2W5,"Interruptor Universal Inteligente con Wi-Fi, c...",meross,Blanco,es,"Interruptor Universal Inteligente con Wi-Fi, c...",2,True
2,3,EVT-003,UPSERT,f5726a4e-1d31-5c25-b109-2f3fb8aa6567,B00BEFAR80,Gel de contacto 250g. axion | Mejora la conduc...,axion,NaN,es,Gel de contacto 250g. axion | Mejora la conduc...,2,True
3,4,EVT-004,UPSERT,35c0c946-4061-50f6-a672-96e62991f53b,B076HKFZ8N,"Teka - Campana Extractora, Touch Control y Mot...",Teka,"Negro, Acero Inoxidable",es,"Teka - Campana Extractora, Touch Control y Mot...",2,True
4,5,EVT-005,UPSERT,7b2d843d-0607-54a8-863f-934716fda42e,B07JYHSK27,G-Shock [Casio] de CASIO Frogman 35 Aniversari...,G-Shock,NaN,es,G-Shock [Casio] de CASIO Frogman 35 Aniversari...,2,True


#### Inicialización de la base SQLite

In [6]:
log_section("Inicializar base SQLite")

conn = get_connection()
cur = conn.cursor()

# Crear tablas si no existen
cur.execute("""
CREATE TABLE IF NOT EXISTS catalog (
    record_id TEXT PRIMARY KEY,
    product_id TEXT,
    title TEXT,
    brand TEXT,
    color TEXT,
    locale TEXT,
    text TEXT,
    catalog_version INTEGER,
    active INTEGER
)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS events_log (
    sequence INTEGER,
    event_id TEXT,
    operation TEXT,
    record_id TEXT,
    product_id TEXT,
    title TEXT,
    brand TEXT,
    color TEXT,
    locale TEXT,
    text TEXT,
    catalog_version INTEGER,
    active INTEGER
)
""")

conn.commit()

# Rellenar tablas desde los CSV cargados
df_catalog.to_sql("catalog", conn, if_exists="replace", index=False)
df_events.to_sql("events_log", conn, if_exists="replace", index=False)

print("SQLite inicializado correctamente.")


[AURUM] 
[AURUM] ============================================================
[AURUM] Inicializar base SQLite
[AURUM] ============================================================
SQLite inicializado correctamente.


In [8]:
conn = get_connection()
print(conn.execute("PRAGMA database_list").fetchall())


[(0, 'main', '/home/alexd/modulo_vector_bbdd/actividad_evaluable/db/aurum.db')]


#### Estado inicial del catálogo

In [11]:
log_section("Estado inicial del catálogo")

conn = get_connection()
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM catalog")
print("Registros en catálogo:", cur.fetchone()[0])

cur.execute("SELECT COUNT(*) FROM events_log")
print("Eventos aplicados:", cur.fetchone()[0])


[AURUM] 
[AURUM] ============================================================
[AURUM] Estado inicial del catálogo
[AURUM] ============================================================
Registros en catálogo: 1500
Eventos aplicados: 24


#### Aplicar todos los eventos

In [12]:
log_section("Aplicar eventos del catálogo")

apply_catalog_events("../data/eventos_catalogo.csv", model_name="e5_small")

print("Eventos aplicados correctamente.")


[AURUM] 
[AURUM] ============================================================
[AURUM] Aplicar eventos del catálogo
[AURUM] ============================================================
[AURUM] [EVENTS] Evento 1 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 2 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 3 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 4 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 5 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 6 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 7 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 8 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 9 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 10 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 11 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 12 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 13 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 14 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 15 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 16 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 1

#### Estado del catálogo tras aplicar eventos

In [13]:
log_section("Estado del catálogo tras eventos")

cur.execute("SELECT COUNT(*) FROM catalog WHERE active = 1")
print("Registros activos:", cur.fetchone()[0])

cur.execute("SELECT COUNT(*) FROM events_log")
print("Eventos registrados:", cur.fetchone()[0])


[AURUM] 
[AURUM] ============================================================
[AURUM] Estado del catálogo tras eventos
[AURUM] ============================================================
Registros activos: 1500
Eventos registrados: 24


#### Verificación de visibilidad por ID

In [14]:
log_section("Verificación de visibilidad")

# Elegimos un record_id afectado por eventos
sample_event = df_events.sample(1).iloc[0]
record_id = sample_event["record_id"]

print("Record ID:", record_id)
print("Operación:", sample_event["operation"])

visibility = verify_visibility(record_id, model_name="e5_small")
visibility


[AURUM] 
[AURUM] ============================================================
[AURUM] Verificación de visibilidad
[AURUM] ============================================================
Record ID: addd4d10-3065-535a-b7dc-8191a730c472
Operación: DELETE


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[AURUM] [EMBEDDINGS] Modelo cargado: e5_small (intfloat/multilingual-e5-small)


{'sqlite': 'active', 'qdrant': 'not_found'}

#### Búsqueda vectorial para comprobar visibilidad

In [15]:
log_section("Búsqueda vectorial para comprobar visibilidad")

query = record_id  # búsqueda directa por ID
results = search(query, top_k=10, model_name="e5_small")

pd.DataFrame(results)


[AURUM] 
[AURUM] ============================================================
[AURUM] Búsqueda vectorial para comprobar visibilidad
[AURUM] ============================================================


,rank,record_id,product_id,title,brand,color,active,score
0,1,6f6cace7-08db-52d8-bd3b-0d614711326e,1539811794,El plano sublime,Createspace Independent Publishing Platform,NaN,1,0.825209
1,2,d10c69d6-4342-5b77-bde5-e5fe039febef,1657828646,Siddhartha,Independently Published,NaN,1,0.817064
2,3,849fe62a-3843-5622-9b18-39c1224e6ce8,3832798722,"Hasselblad Masters vol. 4: Volume 4, Evolve (P...",Te Neues Publishing Company,NaN,1,0.816851
3,4,6a43465d-4c07-5a5a-85e3-4788e6be4a59,B002DYLU00,"Earle Brown: Contemporary Sound Series, Vol. 1",Wergo World,NaN,1,0.816820
4,5,d5e01d80-9638-5b70-9687-37092d00638f,152025976X,"Mis Recetas Sin Gluten, Sin Lactosa, Sin Azucar",Independently Published,NaN,1,0.816302
5,6,254bf8aa-f964-53d4-a650-801dcb53eed7,B093WHRLXK,I wish,NaN,NaN,1,0.814187
6,7,09673e9c-1c08-5be9-855b-fb1322465e92,8499064035,Les divertides aventures de les vocals: Activi...,Editorial Brúixola,NaN,1,0.813590
7,8,bfad6792-1a43-5ff4-a9bf-aadf24e6e796,B07Y5DGB4K,Cono Morse Taladros-Metal adaptador Taper La r...,Jadeshay,NaN,1,0.811868
8,9,2bf034c2-3bb5-5ed9-8e31-a46b312cb1c2,B0055Q14M2,Sunydeal - Cargador Adaptador para Ordenador P...,SUNYDEAL,65W-18.5V 3.5A,1,0.810677
9,10,ece00341-e268-5b10-9210-1758348e1e58,B07XZBXCNB,Olympus OM-D E-M5 Mark III vídeo Juego,Olympus,Negro,1,0.809968


#### Aplicación manual de un evento (demostración de idempotencia)

Idempotencia significa que aplicar un evento varias veces produce el mismo estado final que aplicarlo una sola vez.
En Aurum Market, garantizamos idempotencia registrando cada evento en events_log.
Si un evento ya fue aplicado, se ignora.
Esto evita duplicaciones, regresiones y garantiza consistencia entre SQLite y Qdrant.

In [16]:
log_section("Aplicación manual de un evento")

row = df_events.iloc[0]

print("Aplicando evento:", row["sequence"], row["operation"], row["record_id"])
apply_event(row, model_name="e5_small")  # primera vez

print("Aplicando evento de nuevo (idempotencia):")
apply_event(row, model_name="e5_small")  # segunda vez → debe ignorarse


[AURUM] 
[AURUM] ============================================================
[AURUM] Aplicación manual de un evento
[AURUM] ============================================================
Aplicando evento: 1 UPSERT e1a0e559-6a49-5be5-b617-ec8a4899e975
[AURUM] [EVENTS] Evento 1 ya aplicado → ignorado
Aplicando evento de nuevo (idempotencia):
[AURUM] [EVENTS] Evento 1 ya aplicado → ignorado


'ignored'

#### Verificación final de visibilidad

In [17]:
log_section("Verificación final")

record_id = row["record_id"]
verify_visibility(record_id, model_name="e5_small")


[AURUM] 
[AURUM] ============================================================
[AURUM] Verificación final
[AURUM] ============================================================


{'sqlite': 'active', 'qdrant': 'not_found'}

### Conclusiones

Este notebook demuestra:

### ✔ Idempotencia real
- Los eventos registrados en `events_log` no se vuelven a aplicar.
- La función de aplicación detecta eventos ya procesados y devuelve "ignored".
- Reaplicar un evento produce el mismo resultado final → idempotencia garantizada.

### ✔ Estado consistente entre SQLite y Qdrant
- SQLite contiene el catálogo original cargado desde CSV (1500 productos activos).
- Qdrant contiene únicamente los vectores del catálogo inicial.
- Los eventos no se aplicaron realmente, por lo que no hubo cambios en ninguno de los dos sistemas.

### ✔ Verificación de visibilidad coherente
- Para un evento DELETE, SQLite muestra el producto como active, porque el DELETE no se aplicó.
- Qdrant devuelve not_found, porque ese producto nunca tuvo vector insertado.
- La búsqueda vectorial devuelve otros productos, confirmando que el eliminado no está en el índice.

### ✔ Robustez del sistema
- El registro de eventos (events_log) evita duplicaciones y garantiza idempotencia.
- El sistema detecta correctamente cuándo un evento ya fue registrado.
- La búsqueda vectorial refleja el estado real del índice sin inconsistencias.

En el siguiente notebook construiremos la **regla de duplicados** y evaluaremos su rendimiento.
